# FS-Versioned 3D-TransUNet Experiments

This notebook runs four short fine-tuning experiments from `checkpoints/transunet3d_best.pt`:
- FS v5: train/val/test subsets filtered from `tensors`
- FS v6: train/val/test subsets filtered from `tensors`
- FS v7: train/val/test subsets filtered from `tensors`
- FS v8: convert `gcloud_outputs` -> `tensors_gcloud`, split reproducibly, then train/evaluate

All outputs are written to unique run directories under:
- `checkpoints_fsv5/`
- `checkpoints_fsv6/`
- `checkpoints_fsv7/`
- `checkpoints_fsv8/`


In [1]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from segmentation.config import DEVICE, SEED, MODULE_PATH
from segmentation.plot import plot_region_dice, plot_tissue_dice
from segmentation.fs_experiments import (
    build_v8_split_from_root,
    build_versioned_splits_from_existing_split,
    convert_gcloud_mgz_to_tensors,
    load_fs_version_map,
    run_short_finetune_experiment,
    validate_training_versions,
)
from segmentation.splits import build_or_load_training_split_manifest, split_from_manifest

print(f"device={DEVICE} | seed={SEED}")


device=cuda | seed=1337


In [2]:
BASE_PATH = MODULE_PATH.parent

BASE_CHECKPOINT = BASE_PATH / "checkpoints/transunet3d_best.pt"
FS_VERSION_JSON = BASE_PATH / "fs_version_by_dataset_updated.json"

TENSORS_ROOT = BASE_PATH / "tensors"
TENSORS_ROOT_HCP = BASE_PATH / "tensors" / "convolutional_ar" / "segmentation" / "hcp_filt"
MGZ_ROOT = BASE_PATH / "gcloud_outputs"
TENSORS_ROOT_V8 = BASE_PATH / "tensors_gcloud"

SPLIT_RATIOS = (0.8, 0.1, 0.1)
RANDOM_SEED = int(SEED)

# Set True to build/update tensors_gcloud from gcloud_outputs.
RUN_CONVERSION = False # precomputed
CONVERT_JOBS = -1

TRAIN_OVERRIDES = {
    "epochs": 5,
    "lr_scheduler": "cosine_warmup",
    "warmup_steps": 0,         # auto-adjusted by runner to ~10% of total steps
    "cosine_total_steps": 100,
    "lr": 5e-4
}

if not BASE_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing base checkpoint: {BASE_CHECKPOINT}")
if not FS_VERSION_JSON.exists():
    raise FileNotFoundError(f"Missing version map: {FS_VERSION_JSON}")
if not TENSORS_ROOT_HCP.exists():
    raise FileNotFoundError(f"Missing HCP tensor root: {TENSORS_ROOT_HCP}")


In [3]:
conversion_summary = None
if RUN_CONVERSION:
    conversion_summary = convert_gcloud_mgz_to_tensors(
        gcloud_root=MGZ_ROOT,
        out_root=TENSORS_ROOT_V8,
        n_jobs=CONVERT_JOBS,
    )
    display(pd.DataFrame([conversion_summary]))
else:
    print("Skipping gcloud conversion (RUN_CONVERSION=False)")


Skipping gcloud conversion (RUN_CONVERSION=False)


In [5]:
fs_major_by_dataset = load_fs_version_map(FS_VERSION_JSON)

# Use the exact original training split from checkpoint-derived manifest.
split_manifest = build_or_load_training_split_manifest(force_rebuild=False)
base_split = split_from_manifest(split_manifest, split_key="train_notebook_split")

# v5/v6/v7: partition the existing split by FS major (no re-splitting).
# Unmapped dataset ids (e.g., HCP paths) are treated as FS v5.
versioned_splits = build_versioned_splits_from_existing_split(
    base_split=base_split,
    fs_major_by_dataset=fs_major_by_dataset,
    target_majors=(5, 6, 7),
    default_major_for_unmapped=5,
    error_on_unassigned=True,
)

# Assert exact ID-level equivalence with the original split.
def _norm_paths(paths):
    return {str(Path(p).resolve()) for p in paths}

for split_name in ("train", "val", "test"):
    base_ids = _norm_paths(base_split[f"x_{split_name}"])
    combined_ids = _norm_paths(
        list(versioned_splits[5][f"x_{split_name}"])
        + list(versioned_splits[6][f"x_{split_name}"])
        + list(versioned_splits[7][f"x_{split_name}"])
    )
    if base_ids != combined_ids:
        raise RuntimeError(
            f"Split mismatch for {split_name}: "
            f"base={len(base_ids)} combined={len(combined_ids)} "
            f"overlap={len(base_ids & combined_ids)}"
        )

print("v5/v6/v7 split union matches original training split IDs exactly.")

# v8: all samples in tensors_gcloud are treated as FreeSurfer v8.
split_v8 = build_v8_split_from_root(
    tensors_root=TENSORS_ROOT_V8,
    seed=RANDOM_SEED,
    ratios=SPLIT_RATIOS,
)

# Enforce version validity in training sets.
for major in (5, 6, 7):
    _ = validate_training_versions(
        versioned_splits[major]["x_train"],
        fs_major_by_dataset,
        expected_major=major,
        default_major=(5 if major == 5 else None),
    )
_ = validate_training_versions(
    split_v8["x_train"],
    fs_major_by_dataset,
    expected_major=8,
    default_major=8,
)

split_rows = []
for major in (5, 6, 7):
    s = versioned_splits[major]
    source = f"checkpoint split manifest ({split_manifest['source']['checkpoint_path']})"
    split_rows.append({
        "fs_major": major,
        "n_train": len(s["x_train"]),
        "n_val": len(s["x_val"]),
        "n_test": len(s["x_test"]),
        "source": source,
    })
split_rows.append({
    "fs_major": 8,
    "n_train": len(split_v8["x_train"]),
    "n_val": len(split_v8["x_val"]),
    "n_test": len(split_v8["x_test"]),
    "source": str(TENSORS_ROOT_V8),
})

split_summary_df = pd.DataFrame(split_rows).sort_values("fs_major").reset_index(drop=True)
display(split_summary_df)

v5/v6/v7 split union matches original training split IDs exactly.


,fs_major,n_train,n_val,n_test,source
0,5,720,90,90,checkpoint split manifest (/home/rph/convoluti...
1,6,1592,202,202,checkpoint split manifest (/home/rph/convoluti...
2,7,1096,136,136,checkpoint split manifest (/home/rph/convoluti...
3,8,126,16,16,/home/rph/convolutional_ar/segmentation/tensor...


In [6]:
# train samples: 3405
# val samples:   428
# test samples:  428

In [7]:
# Exact filename parity check vs shared split used in other notebooks.

shared_manifest = build_or_load_training_split_manifest(force_rebuild=False)
shared_split = split_from_manifest(shared_manifest, split_key="train_notebook_split")

current_split = {
    "x_train": list(versioned_splits[5]["x_train"]) + list(versioned_splits[6]["x_train"]) + list(versioned_splits[7]["x_train"]),
    "y_train": list(versioned_splits[5]["y_train"]) + list(versioned_splits[6]["y_train"]) + list(versioned_splits[7]["y_train"]),
    "x_val": list(versioned_splits[5]["x_val"]) + list(versioned_splits[6]["x_val"]) + list(versioned_splits[7]["x_val"]),
    "y_val": list(versioned_splits[5]["y_val"]) + list(versioned_splits[6]["y_val"]) + list(versioned_splits[7]["y_val"]),
    "x_test": list(versioned_splits[5]["x_test"]) + list(versioned_splits[6]["x_test"]) + list(versioned_splits[7]["x_test"]),
    "y_test": list(versioned_splits[5]["y_test"]) + list(versioned_splits[6]["y_test"]) + list(versioned_splits[7]["y_test"]),
}

def _norm_set(paths):
    return {str(Path(p).resolve()) for p in paths}

def _fingerprint(paths):
    import hashlib
    payload = "\n".join(sorted(_norm_set(paths)))
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:16]

for key in ("x_train", "y_train", "x_val", "y_val", "x_test", "y_test"):
    cur = _norm_set(current_split[key])
    ref = _norm_set(shared_split[key])
    if cur != ref:
        only_ref = sorted(ref - cur)
        only_cur = sorted(cur - ref)
        raise AssertionError(
            f"{key} mismatch: ref_only={len(only_ref)} cur_only={len(only_cur)}\n"
            f"ref_only_example={only_ref[:1]}\n"
            f"cur_only_example={only_cur[:1]}"
        )

print("Exact filename parity OK (x/y train/val/test) vs shared training split manifest.")
print("fingerprints:")
for key in ("x_train", "x_val", "x_test"):
    print(f"  {key}: {_fingerprint(current_split[key])}")


Exact filename parity OK (x/y train/val/test) vs shared training split manifest.
fingerprints:
  x_train: 799bb8206d1f0f30
  x_val: 628d4efc9496259b
  x_test: 3c87af96442e2012


In [8]:
# Validate split correctness against original training notebook behavior.

# 1) Raw split IDs (before cache filtering) must match checkpoint split totals.
assert split_summary_df[split_summary_df["fs_major"].isin([5, 6, 7])]["n_train"].sum() == 3408
assert split_summary_df[split_summary_df["fs_major"].isin([5, 6, 7])]["n_val"].sum() == 428
assert split_summary_df[split_summary_df["fs_major"].isin([5, 6, 7])]["n_test"].sum() == 428

# 2) Usable counts after cache filtering must match original run output.
from segmentation.config import build_runtime_cfgs
from segmentation.api import prepare_data_pipeline

combined_split = {
    "x_train_files": list(versioned_splits[5]["x_train"]) + list(versioned_splits[6]["x_train"]) + list(versioned_splits[7]["x_train"]),
    "y_train_files": list(versioned_splits[5]["y_train"]) + list(versioned_splits[6]["y_train"]) + list(versioned_splits[7]["y_train"]),
    "x_val_files": list(versioned_splits[5]["x_val"]) + list(versioned_splits[6]["x_val"]) + list(versioned_splits[7]["x_val"]),
    "y_val_files": list(versioned_splits[5]["y_val"]) + list(versioned_splits[6]["y_val"]) + list(versioned_splits[7]["y_val"]),
    "x_test_files": list(versioned_splits[5]["x_test"]) + list(versioned_splits[6]["x_test"]) + list(versioned_splits[7]["x_test"]),
    "y_test_files": list(versioned_splits[5]["y_test"]) + list(versioned_splits[6]["y_test"]) + list(versioned_splits[7]["y_test"]),
    "group_split_enabled": False,
}

DATA_CFG_CHECK, _, TRAIN_CFG_CHECK, _ = build_runtime_cfgs(
    data_overrides=combined_split,
    model_overrides={},
    train_overrides={"num_workers": 0},
)
pipeline_check = prepare_data_pipeline(DATA_CFG_CHECK, TRAIN_CFG_CHECK)

assert len(pipeline_check.train_ds) == 3405
assert len(pipeline_check.val_ds) == 428
assert (len(pipeline_check.test_ds) if pipeline_check.test_ds is not None else 0) == 428

print("split validation passed: raw=3408/428/428, usable(after cache)=3405/428/428")


cache build/check:   0%|          | 0/3408 [00:00<?, ?it/s]

cache build/check:   0%|          | 0/428 [00:00<?, ?it/s]

cache build/check:   0%|          | 0/428 [00:00<?, ?it/s]

split validation passed: raw=3408/428/428, usable(after cache)=3405/428/428


In [9]:
experiment_plan = {
    5: {"split": versioned_splits[5], "checkpoint_base_dir": Path("checkpoints_fsv5")},
    6: {"split": versioned_splits[6], "checkpoint_base_dir": Path("checkpoints_fsv6")},
    7: {"split": versioned_splits[7], "checkpoint_base_dir": Path("checkpoints_fsv7")},
    8: {"split": split_v8,            "checkpoint_base_dir": Path("checkpoints_fsv8")},
}

results = {}
for fs_major, spec in experiment_plan.items():
    split = spec["split"]
    if len(split["x_train"]) == 0 or len(split["x_test"]) == 0:
        print(f"[skip fsv{fs_major}] need non-empty train and test split")
        continue

    result = run_short_finetune_experiment(
        run_name=f"fsv{fs_major}",
        fs_major=fs_major,
        split=split,
        base_checkpoint_path=BASE_CHECKPOINT,
        checkpoint_base_dir=spec["checkpoint_base_dir"],
        fs_major_by_dataset=fs_major_by_dataset,
        default_major_for_unmapped=(5 if fs_major == 5 else None),
        train_overrides=TRAIN_OVERRIDES,
        seed=RANDOM_SEED,
        device=DEVICE,
    )
    results[fs_major] = result

summary_rows = []
for fs_major, result in sorted(results.items()):
    summary_rows.append({
        "fs_major": fs_major,
        "run_name": result.run_name,
        "n_train": result.n_train,
        "n_val": result.n_val,
        "n_test": result.n_test,
        "warmup_steps": result.warmup_steps,
        "cosine_total_steps": result.cosine_total_steps,
        "best_epoch": result.best_epoch,
        "best_val_ce": result.best_val_ce,
        "final_test_ce": result.final_test_ce,
        "run_dir": str(result.run_dir),
        "checkpoint_path": str(result.checkpoint_path),
    })

results_summary_df = pd.DataFrame(summary_rows).sort_values("fs_major").reset_index(drop=True)
display(results_summary_df)


cache build/check:   0%|          | 0/720 [00:00<?, ?it/s]

cache build/check:   0%|          | 0/90 [00:00<?, ?it/s]

cache build/check:   0%|          | 0/90 [00:00<?, ?it/s]

/home/rph/convolutional_ar/segmentation/segmentation/model.py:107: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(enc_layer, num_layers=transformer_depth)


fsv5 | epoch 001 train:   0%|          | 0/720 [00:00<?, ?it/s]

...

fsv8 | epoch 005 val:   0%|          | 0/16 [00:00<?, ?it/s]

[fsv8] epoch 005 | train_ce=0.0796 | val_ce=0.0803 | lr=1.00e-06 | mb=1 | pc=192 | vram=0.1/14.4GB | t=46.3s
[fsv8] saved best checkpoint -> checkpoints_fsv8/fsv8_20260402_034229/transunet3d_best.pt (epoch=5, val_ce=0.0803)


fsv8 | final test:   0%|          | 0/16 [00:00<?, ?it/s]

test metrics (fast):   0%|          | 0/16 [00:00<?, ?it/s]

,fs_major,run_name,n_train,n_val,n_test,warmup_steps,cosine_total_steps,best_epoch,best_val_ce,final_test_ce,run_dir,checkpoint_path
0,5,fsv5,720,90,90,360,3600,5,0.034929,0.035218,checkpoints_fsv5/fsv5_20260402_015649,checkpoints_fsv5/fsv5_20260402_015649/transune...
1,6,fsv6,1592,202,202,796,7955,5,0.036987,0.034484,checkpoints_fsv6/fsv6_20260402_021927,checkpoints_fsv6/fsv6_20260402_021927/transune...
2,7,fsv7,1096,136,136,547,5470,5,0.039359,0.039579,checkpoints_fsv7/fsv7_20260402_031018,checkpoints_fsv7/fsv7_20260402_031018/transune...
3,8,fsv8,126,16,16,63,630,5,0.080335,0.077117,checkpoints_fsv8/fsv8_20260402_034229,checkpoints_fsv8/fsv8_20260402_034229/transune...


In [ ]:
if not results:
    raise RuntimeError("No completed experiments found.")

majors = sorted(results.keys())
fig, axes = plt.subplots(
    nrows=2,
    ncols=len(majors),
    figsize=(7 * len(majors), 14),
    constrained_layout=True,
)
if len(majors) == 1:
    axes = axes.reshape(2, 1)

for col, fs_major in enumerate(majors):
    bundle = results[fs_major].plot_bundle

    ax_region = axes[0, col]
    plot_region_dice(bundle, ax=ax_region)
    ax_region.set_title(f"FS v{fs_major}: Region Dice")

    ax_tissue = axes[1, col]
    plot_tissue_dice(bundle, ax=ax_tissue)
    ax_tissue.set_title(f"FS v{fs_major}: Tissue Dice")

plot_root = Path("comparison_plots") / time.strftime("%Y%m%d_%H%M%S")
plot_root.mkdir(parents=True, exist_ok=False)
fig_path = plot_root / "fsversion_region_tissue_compare.png"
fig.savefig(fig_path, dpi=200)

print(f"saved comparison figure -> {fig_path}")
results_summary_df.to_csv(plot_root / "results_summary.csv", index=False)
print(f"saved results summary -> {plot_root / 'results_summary.csv'}")


## Notes

- The fine-tuning runner always creates a unique run directory and never overwrites prior runs.
- Each run writes `history.csv`, `sample_metrics.csv`, `region_metrics.csv`, `timing.csv`, and `summary.csv`.
- Region/tissue plots are generated with `plot_region_dice` and `plot_tissue_dice` from `segmentation/plot.py`.
